# Attention

## Queries, Keys and Values

$$
Q = X W_Q, \qquad K = X W_K, \qquad V = X W_V, \qquad W_Q, W_K, W_V \in \mathbb{R}^{d_{\text{model}} \times d_k}.
$$

$$
S_{ts} = \frac{q_t \cdot k_s}{\sqrt{d_k}}, \qquad S = \frac{Q K^\top}{\sqrt{d_k}} \in \mathbb{R}^{T \times T}.
$$

$$
A_{ts} = \frac{\exp(S_{ts})}{\sum_{r=1}^{T} \exp(S_{tr})}, \qquad A_{ts} \ge 0, \quad \sum_{s=1}^{T} A_{ts} = 1.
$$

$$
o_t = \sum_{s=1}^{T} A_{ts}\, v_s, \qquad O = A V \in \mathbb{R}^{T \times d_k}.
$$

## A Small Example

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

vocabulary = ["alice", "saw", "the", "white", "rabbit"]
word_to_id = {word: i for i, word in enumerate(vocabulary)}

d_model = 4                                      # size of the word vectors
rng = np.random.default_rng(3)
embeddings = np.round(rng.normal(size=(len(vocabulary), d_model)), 1)     # one row per word (random, rounded to be readable)

sentence = ["alice", "saw", "the", "white", "rabbit"]
X = embeddings[[word_to_id[w] for w in sentence]]                          # T x d_model, one row per token
pd.DataFrame(X, index=sentence, columns=[f"x{j + 1}" for j in range(d_model)])

,x1,x2,x3,x4
alice,2.0,-2.6,0.4,-0.6
saw,-0.5,-0.2,-2.0,-0.2
the,-0.9,3.3,0.2,-0.4
white,-0.3,-0.7,-1.1,-0.4
rabbit,0.5,-0.2,1.0,-0.2


In [2]:
d_k = 2                                          # size of the queries, keys and values
W_Q = np.round(rng.normal(size=(d_model, d_k)), 1)
W_K = np.round(rng.normal(size=(d_model, d_k)), 1)
W_V = np.round(rng.normal(size=(d_model, d_k)), 1)

Q, K, V = X @ W_Q, X @ W_K, X @ W_V              # each T x d_k
print("shapes:", X.shape, W_Q.shape, "->", Q.shape)
pd.concat({"queries Q": pd.DataFrame(Q, index=sentence).round(2),
           "keys K": pd.DataFrame(K, index=sentence).round(2),
           "values V": pd.DataFrame(V, index=sentence).round(2)}, axis=1)

shapes: (5, 4) (4, 2) -> (5, 2)


queries Q       keys K       values V      
               0     1      0     1        0     1
alice      -2.52  4.68   2.24  2.60    -2.64  7.36
saw        -0.08 -1.59  -1.54 -1.78     1.08 -1.34
the         0.85 -2.78  -2.65 -2.05    -0.56 -5.93
white      -0.89 -0.53  -0.34 -1.03     1.37  0.56
rabbit     -0.68  1.41   0.96  1.02    -0.82  1.76

In [3]:
scores = Q @ K.T / np.sqrt(d_k)                  # T x T: scores[t, s] = q_t . k_s / sqrt(d_k)
pd.DataFrame(scores, index=sentence, columns=sentence).round(2)

,alice,saw,the,white,rabbit
alice,4.61,-3.15,-2.06,-2.80,1.66
saw,-3.05,2.09,2.45,1.18,-1.20
the,-3.76,2.57,2.44,1.82,-1.43
white,-2.38,1.64,2.44,0.60,-0.99
rabbit,1.52,-1.03,-0.77,-0.86,0.56


In [4]:
def softmax_rows(scores):
    shifted = scores - scores.max(axis=1, keepdims=True)       # subtracting the row maximum does not change the result but avoids overflow
    exp_scores = np.exp(shifted)
    return exp_scores / exp_scores.sum(axis=1, keepdims=True)

A = softmax_rows(scores)                         # attention weights, T x T
attention_table = pd.DataFrame(A, index=sentence, columns=sentence)
print("row sums:", A.sum(axis=1))
attention_table.round(2)

row sums: [1. 1. 1. 1. 1.]


,alice,saw,the,white,rabbit
alice,0.95,0.00,0.00,0.00,0.05
saw,0.00,0.35,0.50,0.14,0.01
the,0.00,0.42,0.37,0.20,0.01
white,0.00,0.27,0.61,0.10,0.02
rabbit,0.60,0.05,0.06,0.06,0.23


In [5]:
O = A @ V                                        # T x d_k: every output is a weighted average of the value vectors
t = sentence.index("rabbit")
manual = sum(A[t, s] * V[s] for s in range(len(sentence)))
print("output for 'rabbit' (A @ V):", O[t].round(3))
print("weighted sum of the values :", manual.round(3))
pd.DataFrame(O, index=sentence).round(2)

output for 'rabbit' (A @ V): [-1.691  4.456]
weighted sum of the values : [-1.691  4.456]


,0,1
alice,-2.54,7.06
saw,0.27,-3.31
the,0.51,-2.63
white,0.06,-3.83
rabbit,-1.69,4.46


In [7]:
def self_attention(X, W_Q, W_K, W_V):
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V
    A = softmax_rows(Q @ K.T / np.sqrt(K.shape[1]))
    return A @ V, A

output, weights = self_attention(X, W_Q, W_K, W_V)
print("same result as above:", np.allclose(output, O) and np.allclose(weights, A))

same result as above: True


### Why Divide by $\sqrt{d_k}$?

### Attention Does Not Know the Word Order

## A Tiny Trained Example

In [10]:
import itertools
import torch
import torch.nn as nn

# tiny vocabulary; "?" marks the position whose answer we want
characters = ["queen", "rabbit", "alice", "cat", "tea"]        # hidden rule: the earlier in this list, the more important
vocab_small = ["?"] + characters
small_id = {w: i for i, w in enumerate(vocab_small)}

# every sentence: 3 characters (repetitions allowed), followed by "?"; the target is the most important of the 3
sentences = [list(p) + ["?"] for p in itertools.product(characters, repeat=3)]
targets = [min(s[:3], key=characters.index) for s in sentences]
X_ids = torch.tensor([[small_id[w] for w in s] for s in sentences])
y_ids = torch.tensor([characters.index(t) for t in targets])

torch.manual_seed(0)
perm = torch.randperm(len(sentences))
train_idx, test_idx = perm[:90], perm[90:]
print(len(sentences), "possible sentences,", len(train_idx), "for training,", len(test_idx), "for testing")
for k in range(3):
    print(sentences[k], "->", targets[k])

125 possible sentences, 90 for training, 35 for testing
['queen', 'queen', 'queen', '?'] -> queen
['queen', 'queen', 'rabbit', '?'] -> queen
['queen', 'queen', 'alice', '?'] -> queen


In [11]:
class SingleHeadAttention(nn.Module):
    def __init__(self, d_model, d_k):
        super().__init__()
        self.W_Q = nn.Linear(d_model, d_k, bias=False)
        self.W_K = nn.Linear(d_model, d_k, bias=False)
        self.W_V = nn.Linear(d_model, d_k, bias=False)
        self.d_k = d_k

    def forward(self, X):                                        # X: (batch, T, d_model)
        Q, K, V = self.W_Q(X), self.W_K(X), self.W_V(X)
        A = torch.softmax(Q @ K.transpose(1, 2) / self.d_k ** 0.5, dim=-1)       # (batch, T, T)
        return A @ V, A


class AttentionClassifier(nn.Module):
    def __init__(self, vocab_size, n_classes, d_model=8, d_k=8):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.attention = SingleHeadAttention(d_model, d_k)
        self.output = nn.Linear(d_k, n_classes)

    def forward(self, word_ids):
        out, A = self.attention(self.embedding(word_ids))
        return self.output(out[:, -1]), A                        # the answer is read from the position of "?"

In [12]:
torch.manual_seed(0)
model = AttentionClassifier(len(vocab_small), len(characters))
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
loss_function = nn.CrossEntropyLoss()

for epoch in range(300):
    optimizer.zero_grad()
    logits, _ = model(X_ids[train_idx])
    loss = loss_function(logits, y_ids[train_idx])
    loss.backward()
    optimizer.step()

with torch.no_grad():
    for name, idx in [("training", train_idx), ("test", test_idx)]:
        logits, _ = model(X_ids[idx])
        print(f"{name} accuracy: {(logits.argmax(dim=1) == y_ids[idx]).float().mean().item():.2f}")
    print(f"final training loss: {loss.item():.4f}")

training accuracy: 1.00
test accuracy: 1.00
final training loss: 0.0001


In [13]:
def show_attention(words):
    ids = torch.tensor([[small_id[w] for w in words]])
    with torch.no_grad():
        logits, A = model(ids)
    print(" ".join(words), "->", characters[logits.argmax().item()])
    print("   attention of '?':", [(w, round(float(a), 2)) for w, a in zip(words, A[0, -1])])

for words in [sentences[k] for k in test_idx[:3]] + [["alice", "cat", "tea", "?"], ["tea", "tea", "alice", "?"], ["rabbit", "tea", "rabbit", "?"]]:
    show_attention(words)

rabbit alice queen ? -> queen
   attention of '?': [('rabbit', 0.0), ('alice', 0.0), ('queen', 1.0), ('?', 0.0)]
queen alice cat ? -> queen
   attention of '?': [('queen', 1.0), ('alice', 0.0), ('cat', 0.0), ('?', 0.0)]
rabbit alice tea ? -> rabbit
   attention of '?': [('rabbit', 1.0), ('alice', 0.0), ('tea', 0.0), ('?', 0.0)]
alice cat tea ? -> alice
   attention of '?': [('alice', 1.0), ('cat', 0.0), ('tea', 0.0), ('?', 0.0)]
tea tea alice ? -> alice
   attention of '?': [('tea', 0.0), ('tea', 0.0), ('alice', 1.0), ('?', 0.0)]
rabbit tea rabbit ? -> rabbit
   attention of '?': [('rabbit', 0.5), ('tea', 0.0), ('rabbit', 0.5), ('?', 0.0)]


In [14]:
# how much attention does "?" pay to the most important character? (all sentences)
with torch.no_grad():
    _, A_all = model(X_ids)
is_target = torch.tensor([[float(w == targets[k]) for w in sentences[k]] for k in range(len(sentences))])
weight_on_target = (A_all[:, -1] * is_target).sum(dim=1)          # attention on all positions that hold the most important word
print(f"average attention on the most important character: {weight_on_target.mean():.2f} (uniform attention would give about 0.25)")

# the score q_? . k_w / sqrt(d_k) of every word as a key, for the query of "?"
with torch.no_grad():
    attn = model.attention
    E_words = model.embedding.weight
    query = attn.W_Q(E_words[small_id["?"]])
    key_scores = (attn.W_K(E_words) @ query) / attn.d_k ** 0.5
print(pd.Series(key_scores.numpy().astype(float), index=vocab_small).round(1).to_frame("score").T)

average attention on the most important character: 1.00 (uniform attention would give about 0.25)
          ?  queen  rabbit  alice  cat   tea
score -20.2   20.9     3.3   -2.6 -9.6 -15.3


## Limitations

## Summary

## Exercises